In [1]:
import pandas as pd
import altair as alt

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"
ufo = pd.read_csv(url, header=None)

ufo.columns = ['datetime', 'city', 'state', 'country', 'shape', 'duration_seconds',
               'duration', 'comments', 'date_posted', 'latitude', 'longitude']

ufo['datetime'] = pd.to_datetime(ufo['datetime'], format='%m/%d/%Y %H:%M')
ufo['year'] = ufo['datetime'].dt.year

In [2]:
shapes = ['light', 'triangle', 'circle', 'fireball', 'sphere', 'disk']
ufo_shapes = ufo[ufo['shape'].isin(shapes)]
ufo_shapes = ufo_shapes[(ufo_shapes['year'] >= 1990) & (ufo_shapes['year'] <= 2013)]

yearly = ufo_shapes.groupby(['year', 'shape']).size().reset_index(name='count')

selection = alt.selection_point(fields=['shape'], bind='legend')

chart1 = alt.Chart(yearly).mark_line().encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('count:Q', title='Number of Sightings'),
    color=alt.Color('shape:N', title='Shape'),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=['year', 'shape', 'count']
).add_params(selection).properties(title='UFO Sightings per Year by Shape')

chart1

alt.Chart(...)

In [3]:
us = ufo[ufo['country'] == 'us']
states = us['state'].value_counts().head(15).reset_index()
states.columns = ['state', 'count']

chart2 = alt.Chart(states).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Sightings'),
    y=alt.Y('state:N', title='State', sort='-x'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='purples')),
    tooltip=['state', 'count']
).properties(title='Top 15 States for UFO Sightings')

chart2

alt.Chart(...)

In [4]:
chart1.properties(width='container').save("ufo_chart1.json")
chart2.properties(width='container').save("ufo_chart2.json")